# 00 データと基準 — 日射予報だけで、どこまで当たるか

> 🔒 すべて**合成データ**です。⏱ 所要時間: 説明 15分 ＋ 実行 約1分

## ゴール
- 日次の**太陽光発電量**の合成データが、どんな構造か（天気・季節・**見えない状態**）を理解する。
- **予報のリード**（何日前に出した予報か）で、予測の難しさが変わることを確かめる。
- 学習を使わない**基準**（持続予報・平年値・予報×0.85・直線回帰）を作り、**深層学習が超えるべき水準**を知る。

## なぜ基準から始めるのか
深層学習は「何でも当てる」わけではありません。**単純な方法にどれだけ勝てるか**で価値を測ります。時系列予測では特に、「昨日の値をそのまま使う」「予報を換算する」といった単純な方法が、**驚くほど強い**ことがあります。

## データの作り方
```
発電量 = 0.85 × 真の日射量 × 温度の低下 × (1 − よごれ) × (1 + ノイズ3%)
```
- **温度の低下**: パネルが熱いと効率が下がる（25℃を超えた分に比例）。気温から推測できる。
- **よごれ**（隠れた状態）: 雨が降らない日が続くと少しずつ増え、雨の日に洗い流される。**直接は観測できない**が、**過去の天気の並び**から推測できる。
- **日射の予報**: 先の日（リードが長い）ほど不正確で、平年並みに寄っていく。

この「見えない部分」があるので、**過去の並びを読めるモデル**（のちのフェーズの系列モデル）に伸びしろがあるかを、確かめられます。

## キーワード
- **リード L**: 予報を出してから対象日までの日数。L=1 は前日の夜に出した予報。
- **持続予報**: L日前の値をそのまま予測にする。時系列予測の最も基本的な基準。
- **平年値（気候値）**: 同じ日付の、過去の平均。
- **MAE**: 平均絶対誤差（kWh/日）。小さいほど良い。
- **スキル**: `1 − MAE/MAE(持続予報)`。0=持続予報と同じ、1=完璧。
- **時間順の分割**: 学習（過去10年）→ 検証（次の2年）→ テスト（最後の2年）。**未来を学習に使わない**。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import run_phase0
from tsf_synth import make_series, split_series

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
pd.set_option("display.width", 200)

df = make_series(1_000_000)
sp = split_series(df)
run_phase0.e1()

**出力の見方**:
- 系列は5114日（約14年）で、学習3652日・検証730日・テスト732日。**時間順**に並んでいます。
- 発電量の平均は約8 kWh/日。晴れ37%・曇り33%・雨30%。
- **よごれ**は平均3.5%、最大20%（上限）。**温度による低下**は平均2%、最大9.6%。どちらも**小さいが、系統的**な効果です。
- 予報の誤差（MAE、kWh換算）は、リード1で0.92、リード5で3.03と、**先の日ほど約3倍悪い**。

In [ ]:
d = df.iloc[3000:3365]
fig, ax = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
ax[0].plot(d["date"], d["gen_kwh"], label="発電量（観測）", lw=1)
ax[0].plot(d["date"], 0.85 * d["fc1"], label="予報（L=1）×0.85", lw=1, alpha=0.8)
ax[0].set_ylabel("kWh/日"); ax[0].legend(ncol=2)
ax[1].plot(d["date"], 100 * d["soiling"], color="tab:brown"); ax[1].set_ylabel("よごれ（％）\n隠れた状態")
ax[2].plot(d["date"], 100 * (1 - d["derate"]), color="tab:red"); ax[2].set_ylabel("温度による低下（％）")
plt.tight_layout(); plt.show()

**出力の見方**: 1年分です。上段は発電量（青）と、予報を換算したもの（橙）。ほぼ重なりますが、**ずれ**があります。
- 中段の**よごれ**は、晴れ・曇りが続くとのこぎり状に増え、雨の日にがくっと下がります。**実際の手法には見えない**線です。
- 下段の**温度による低下**は、夏に大きい。
- 上段のずれ（橙が青より高い）は、**この2つの効果**（と3%のノイズ）が原因の一部です。

In [ ]:
res = run_phase0.e2()
piv = res.pivot(index="手法", columns="リードL", values="MAE")
order = ["持続予報（L日前の値）", "持続予報（7日平均）", "平年値（日付ごとの平均）", "予報×0.85（学習なし）", "予報の直線回帰", "上限1（真の日射量×0.85）", "上限2（温度・よごれまで知っている）"]
piv.loc[order]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name in order:
    g = res[res["手法"] == name]
    ax[0].plot(g["リードL"], g["MAE"], "o-", label=name)
ax[0].set_xlabel("リード L（日）"); ax[0].set_ylabel("検証のMAE（kWh/日）"); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)
ax[0].set_title("基準の比較（10世界の平均）")
g = res[res["手法"] == "予報の直線回帰"]
ax[1].bar(g["リードL"], g["持続予報に対するスキル"]); ax[1].set_xlabel("リード L（日）"); ax[1].set_ylabel("持続予報に対するスキル")
ax[1].set_title("直線回帰のスキル")
plt.tight_layout(); plt.show()

**出力の見方**（10世界の平均MAE）:
- **持続予報**は、L=1で3.49と**大きな誤差**（天気は毎日変わるため）。リードが長いほど悪くなる（L=5で4.60）。
- **7日平均・平年値**は、L=1では持続予報より悪い（3.61・3.60）が、**L≥2では良い**（平年値3.60はLによらず一定）。「昨日の値」の価値は、リードが長いと薄れ、**平年値に負けます**。
- **予報×0.85**（学習なし）は、L=1で **1.06**と圧倒的。**予報という新しい情報**が最も効きます。
- **予報の直線回帰**は、L=1で **0.875**（直線をデータに当てはめただけで、2割ほど改善）。L=5では2.71と、予報が不正確になるので、平年値（3.60）に近づく。
- **上限1**（真の日射量×0.85、予報が完璧）は **0.636**。それでも0には届かない: **温度とよごれを知らない**ため。
- **上限2**（温度・よごれまで知っている）は **0.194**。残るのはノイズ3%だけ。
- つまり、L=1で、直線回帰（0.875）と上限2（0.194）の間に**大きな伸びしろ**があります。ただし、**予報の誤差**（上限1との差、0.875→0.636）と、**隠れた効果**（上限1→上限2、0.636→0.194）の2つが混ざっています。深層学習が詰められるのは、主に後者（温度・よごれを過去から推測する部分）の一部だけで、**予報の誤差は詰められません**。

In [ ]:
from tsf_eval import linear_fc, breakdown
b = pd.concat([breakdown(linear_fc(sp["train"], sp["val"], 1), sp["val"], "weather"),
               breakdown(linear_fc(sp["train"], sp["val"], 1), sp["val"], "season")]).reset_index(drop=True)
b

**出力の見方**（直線回帰・L=1・1つの世界の検証期間）:
- **天気別**: 晴れ（bias −0.49）は低めに、雨（+0.59）は高めに予測。**直線1本**では、晴れと雨の違いを捉えきれていません。
- **季節別**: 夏のMAEが最大（1.26）、冬が最小（0.63）。**夏は発電量が大きく、温度による低下も大きい**ため。
- こうした**系統的なずれ**は、非線形な手法やより多くの情報（気温・過去の天気）で改善できる余地です。

## 試してみよう（演習）
1. **隠れた状態を消す**: `make_series(1_000_000, soiling_rate=0.0)` で系列を作り、`oracle_physical` と `oracle_full` のMAEの差がどう変わるか見てください。分かるのは、よごれが上限の差にどれだけ効いているかです。
2. **温度の影響を消す**: `temp_coef=0.0` にして同様に比べてください。
3. **リードの違い**: 上の図で、直線回帰が平年値と並ぶのは、おおよそ何日先か読み取ってください。分かるのは、予報が役に立つ期間の目安です。
4. **学習期間を短くする**: `split_series(df, train_years=1)` を作り、直線回帰を当てはめ直して、MAEがどう変わるか見てください（直線は2つのパラメータだけなので、1年でもほぼ同じはずです）。

## まとめ
- 日射**予報**が最も効く情報。**予報×0.85（学習なし）**でL=1のMAE 1.06、直線回帰で 0.875（持続予報 3.49 の約1/4）。
- 持続予報は、天気が毎日変わる系列では**弱い**。リードが長いと平年値に負ける。
- 予報が完璧でも、温度とよごれを知らないと **0.636** が限界。知っていれば **0.194**。この差が、**過去の並びを読むモデル**の伸びしろ（上限）。
- **深層学習が、直線回帰（0.875）をどこまで超えられるか**が、これからの問いです。

次の Phase1 では、**ラグ特徴量と気温を使った古典的な機械学習**（リッジ回帰・ランダムフォレスト・勾配ブースティング）で、基準を超えられるかを調べます。